In [1]:
import collections
import logging
import os
import pathlib
import re
import string
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

import tensorflow_datasets as tfds
import tensorflow_text as text
import tensorflow as tf

In [2]:
logging.getLogger('tensorflow').setLevel(logging.ERROR)  # suppress warnings

In [3]:
from pathlib import Path


def load_data(path) :
    path = Path(path)  # Convert string to Path object
    text = path.read_text(encoding='utf-8')

    lines = text.splitlines( )
    pairs = [ line.split('\t') for line in lines ]

    inp = [ inp for targ , inp in pairs ]
    targ = [ targ for targ , inp in pairs ]

    return inp , targ


def load_data_as_dataset(path) :
    targ , inp = load_data(path)
    dataset = tf.data.Dataset.from_tensor_slices((inp , targ))
    return dataset

In [4]:
train_examples = load_data_as_dataset('translation_train.txt')


In [5]:
val_examples = load_data_as_dataset('translation_val.txt')


In [6]:
for dyu_examples , fr_examples in train_examples.batch(3).take(1) :
    for fr in fr_examples.numpy( ) :
        print(fr.decode('utf-8'))

    print( )

    for dyu in dyu_examples.numpy( ) :
        print(dyu.decode('utf-8'))

Il boit de l’eau.
Il se plaint toujours.
Quoi ? Quelque chose.

A bi ji min na
A le dalakolontɛ lon bɛ.
Mun? Fɛn dɔ.


2024-07-02 17:19:38.386852: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [ ]:
model_name = "ted_hrlr_translate_dyu_fr_converter"
tokenizers = tf.saved_model.load(model_name)


In [ ]:
[ item for item in dir(tokenizers.fr) if not item.startswith('_') ]

In [24]:
for fr in fr_examples.numpy( ) :
    print(fr.decode('utf-8'))

Il boit de l’eau.
Il se plaint toujours.
Quoi ? Quelque chose.


In [25]:
encoded = tokenizers.fr.tokenize(fr_examples)

for row in encoded.to_list( ) :
    print(row)

[2, 59, 19, 486, 56, 29, 52, 252, 12, 3]
[2, 59, 88, 33, 299, 298, 302, 12, 3]
[2, 282, 17, 433, 223, 12, 3]


In [26]:
round_trip = tokenizers.fr.detokenize(encoded)
for line in round_trip.numpy( ) :
    print(line.decode('utf-8'))

il boit de l ’ eau .
il se plaint toujours .
quoi ? quelque chose .


In [27]:
tokens = tokenizers.fr.lookup(encoded)
tokens

<tf.RaggedTensor [[b'[START]', b'il', b'b', b'##oit', b'de', b'l', b'\xe2\x80\x99', b'eau',
  b'.', b'[END]']                                                         ,
 [b'[START]', b'il', b'se', b'p', b'##la', b'##int', b'toujours', b'.',
  b'[END]']                                                            ,
 [b'[START]', b'quoi', b'?', b'quelque', b'chose', b'.', b'[END]']]>

In [28]:
def tokenize_pairs(dyu , fr) :
    dyu = tokenizers.dyu.tokenize(dyu)
    # Convert from ragged to dense, padding with zeros.
    dyu = dyu.to_tensor( )

    fr = tokenizers.fr.tokenize(fr)
    # Convert from ragged to dense, padding with zeros.
    fr = fr.to_tensor( )
    return dyu , fr

In [29]:
BUFFER_SIZE = 20000
BATCH_SIZE = 64

In [ ]:
def make_batches(ds) :
    return (
            ds
            .cache( )
            .shuffle(BUFFER_SIZE)
            .batch(BATCH_SIZE)
            .map(tokenize_pairs , num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))


train_batches = make_batches(train_examples)
val_batches = make_batches(val_examples)

In [1]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
splits = { 'train' : 'data/train-00000-of-00001.parquet' , 'validation' : 'data/validation-00000-of-00001.parquet' , 'test' : 'data/test-00000-of-00001.parquet' }
df = pd.read_parquet("hf://datasets/uvci/Koumankan_mt_dyu_fr/" + splits[ "train" ])

In [2]:
df.head( )

,ID,translation
0,ID_18897661270129,"{'dyu': 'A bi ji min na', 'fr': 'Il boit de l’..."
1,ID_18479132727846,"{'dyu': 'A le dalakolontɛ lon bɛ.', 'fr': 'Il ..."
2,ID_18164131280307,"{'dyu': 'Mun? Fɛn dɔ.', 'fr': 'Quoi ? Quelque ..."
3,ID_18344573728152,"{'dyu': 'O bɛ bi bɔra fo Gubeta.', 'fr': 'Tous..."
4,ID_18127342282717,"{'dyu': 'A ale lo bi da bugɔ la!', 'fr': 'Ah !..."


In [7]:
df[ 'dyu' ] = df[ 'translation' ].apply(lambda x : x[ 'dyu' ])
df[ 'fr' ] = df[ 'translation' ].apply(lambda x : x[ 'fr' ])

In [8]:
df

,ID,translation,dyu,fr
0,ID_18897661270129,"{'dyu': 'A bi ji min na', 'fr': 'Il boit de l’...",A bi ji min na,Il boit de l’eau.
1,ID_18479132727846,"{'dyu': 'A le dalakolontɛ lon bɛ.', 'fr': 'Il ...",A le dalakolontɛ lon bɛ.,Il se plaint toujours.
2,ID_18164131280307,"{'dyu': 'Mun? Fɛn dɔ.', 'fr': 'Quoi ? Quelque ...",Mun? Fɛn dɔ.,Quoi ? Quelque chose.
3,ID_18344573728152,"{'dyu': 'O bɛ bi bɔra fo Gubeta.', 'fr': 'Tous...",O bɛ bi bɔra fo Gubeta.,Tous sortent excepté Gubetta.
4,ID_18127342282717,"{'dyu': 'A ale lo bi da bugɔ la!', 'fr': 'Ah !...",A ale lo bi da bugɔ la!,Ah ! c’est lui… il sonne…
...,...,...,...,...
8060,ID_17804695917936,"{'dyu': 'Alu a yi n ka yanmariya kɔnɔ.', 'fr':...",Alu a yi n ka yanmariya kɔnɔ.,Allez… attendez mes ordres.
8061,ID_19488957919412,"{'dyu': 'O fatura aksidan ni na.', 'fr': 'Ils ...",O fatura aksidan ni na.,Ils ont péri dans l'accident.
8062,ID_18268594920535,"{'dyu': 'An! Ni tericɛ fariman ni.', 'fr': 'Ah...",An! Ni tericɛ fariman ni.,Ah ! ce brave ami !
8063,ID_17525560921507,"{'dyu': 'Sen bi dougouma', 'fr': 'À bas les pa...",Sen bi dougouma,À bas les pattes !


In [9]:
df.drop([ 'ID' , 'translation' ] , axis=1 , inplace=True)

In [10]:
df

,dyu,fr
0,A bi ji min na,Il boit de l’eau.
1,A le dalakolontɛ lon bɛ.,Il se plaint toujours.
2,Mun? Fɛn dɔ.,Quoi ? Quelque chose.
3,O bɛ bi bɔra fo Gubeta.,Tous sortent excepté Gubetta.
4,A ale lo bi da bugɔ la!,Ah ! c’est lui… il sonne…
...,...,...
8060,Alu a yi n ka yanmariya kɔnɔ.,Allez… attendez mes ordres.
8061,O fatura aksidan ni na.,Ils ont péri dans l'accident.
8062,An! Ni tericɛ fariman ni.,Ah ! ce brave ami !
8063,Sen bi dougouma,À bas les pattes !


In [11]:
# Save dyu column to a txt file
df[ 'dyu' ].to_csv('dyu_texts_plain.txt' , index=False , header=False)

In [1]:
from transformers import NllbTokenizer , AutoModelForSeq2SeqLM , AutoConfig

In [2]:
# Load model and tokenizer

model_load_name = "jonathansuru/dioula_saved_model"
model = AutoModelForSeq2SeqLM.from_pretrained(model_load_name)
tokenizer = NllbTokenizer.from_pretrained(model_load_name)

/Users/jonathansuru/anaconda3/envs/LearnHub/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/jonathansuru/anaconda3/envs/LearnHub/lib/python3.11/site-packages/transformers/utils/generic.py:311: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(
/Users/jonathansuru/anaconda3/envs/LearnHub/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [3]:

# Define translate function

def translate(text , src_lang="dyu" , tgt_lang='fra_Latn' , a=16 , b=1.5 , max_input_length=1024 , **kwargs) :
    tokenizer.src_lang = src_lang
    tokenizer.tgt_lang = tgt_lang
    inputs = tokenizer(text , return_tensors='pt' , padding=True , truncation=True , max_length=max_input_length)
    result = model.generate(
            **inputs.to(model.device) ,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang) ,
            max_new_tokens=int(a + b * inputs.input_ids.shape[ 1 ]) ,
            **kwargs
    )
    #print(inputs.input_ids.shape[1], result.shape[1])
    return tokenizer.batch_decode(result , skip_special_tokens=True)


In [11]:

t = "Cɛ a tun bɔnilo a ma"
print(translate(t , 'dyu_Latn' , 'fra_Latn'))

["L'homme à qui il reprochait d'être aimé."]


In [ ]:
https: // cointegrated.medium.com / how - to - fine - tune - a - nllb - 200 - model -
for -translating - a - new - language - a37fc706b865

In [ ]:
https: // github.com / rfeinberg3 / Hittite_English_Translation_w - NLLB / blob / main / jupyter / nllb_hittite_to_english_finetune.ipynb